In [4]:
# Import packages
import os
import re
import sys
import glob
import toml
import shutil
import textwrap
import subprocess

import numpy as np
import pandas as pd

from pathlib import Path
from astropy.io import fits
from importlib import reload
from astropy.table import Table
from scipy.interpolate import interp1d
from joebvp import cfg, utils, VPmeasure

In [5]:
# Register directories
notebook_directory = Path().resolve()
base_directory = notebook_directory / "G130M1291"
python_cmd = sys.executable
model_file = "/Users/billyli/Documents/BillyJessExploratory/powr-OB-lmc-vd3.nc"
runner_path = "/Users/billyli/Documents/BillyJessExploratory/stellar_template_fitting_runner.py"

In [6]:
# Generate terminal commands
directories = sorted([d for d in os.listdir(base_directory) if d.startswith("G130M")])

def replace_string(path, dataset_name):
    segment = """lt_xspec [path][dataset_name]_x1dsum.fits
cd [path] && lt_continuumfit --redshift 0.00087 [dataset_name]_x1dsum.fits [dataset_name]_continuumfit.fits
cd [path] && pyigm_igmguesses [dataset_name]_continuumfit.fits -o [dataset_name]_x1dfits_model.json
cd [path] && pyigm_fitdla --out_file [dataset_name]_xfitdla.json [dataset_name]_continuumfit.fits 0"""
    modified_segment = segment.replace("[dataset_name]", dataset_name)
    modified_segment = modified_segment.replace("[path]", path)
    print(modified_segment)
    
for subdir in directories:
    path = base_directory / subdir
    for file in os.listdir(path):
        if file.endswith("_x1dsum.fits"):
            dataset_name = file.replace("_x1dsum.fits", "")
            path = str(path.resolve()) + "/"
            replace_string(path, dataset_name)
            print("\r")

lt_xspec /Users/billyli/Documents/float-for-morrow/LMC/N11-ELS-018/G130M1291/G130M10/leal2c010_x1dsum.fits
cd /Users/billyli/Documents/float-for-morrow/LMC/N11-ELS-018/G130M1291/G130M10/ && lt_continuumfit --redshift 0.00087 leal2c010_x1dsum.fits leal2c010_continuumfit.fits
cd /Users/billyli/Documents/float-for-morrow/LMC/N11-ELS-018/G130M1291/G130M10/ && pyigm_igmguesses leal2c010_continuumfit.fits -o leal2c010_x1dfits_model.json
cd /Users/billyli/Documents/float-for-morrow/LMC/N11-ELS-018/G130M1291/G130M10/ && pyigm_fitdla --out_file leal2c010_xfitdla.json leal2c010_continuumfit.fits 0



# Didn't use PoWR
for fits_path in sorted(base_directory.rglob("*_x1dsum.fits")):
    
    dataset = fits_path.stem.replace("_x1dsum", "")
    cwd_directory = fits_path.parent

    problem = notebook_directory.name.lower()
    target = notebook_directory.name.upper()

    spec_file = fits_path.name
    
    wave_min = 1135
    wave_max = 1425

    toml_content = textwrap.dedent(f"""
    [target_info]
    problem    = "{problem}"
    target     = "{target}"
    spec_file  = "{spec_file}"
    model_file = "{model_file}"
    instrument = "cos_g130m"

    [include_wave_range]
    wave_range = [ {wave_min}, {wave_max},]

    [exclude_wave_ranges]
    dla = [[1180, 1240,] ]
    gap = [[1270, 1290,] ]

    [vshift]
    min = 50
    max = 550
    delta = 50

    [vsini]
    min = 50
    max = 350
    delta = 50

    [exclude_vranges.ions]
    linelist = "ISM"
    vrange = [-25, 300]
    drop_species = []

    [target_info.plot_ranges_kwargs.all_merged]
    color = "gainsboro"
    """).lstrip()

    tmp_toml = cwd_directory / f"{problem}.toml"
    tmp_toml.write_text(toml_content)

    subprocess.run([python_cmd, runner_path, str(tmp_toml)], check = True, cwd = cwd_directory)

    csv_path = cwd_directory / f"{problem}.csv"
    model_continuum = Table.read(str(csv_path), format = "csv")
    original = Table.read(fits_path, hdu = 1)

    wave = model_continuum['wave'].data
    flux = model_continuum['flux'].data
    err = model_continuum['err'].data
    wave_model = model_continuum['wave'].data
    best_fit_spec = model_continuum['best_fit_spec'].data

    continuumfit_path = cwd_directory / f"{dataset}_continuumfit_original.fits"
    with fits.open(continuumfit_path, mode = 'readonly') as hdul:
        new_hdul = fits.HDUList([h.copy() for h in hdul])

    flux_original = new_hdul[0].data
    wave_original = new_hdul[2].data
    continuum_hdu = new_hdul[3]
    
    continuum_original = continuum_hdu.data
    
    mask = (wave_original >= wave_min) & (wave_original <= wave_max)
    flux_original_masked = flux_original[mask]

    normalization_factor = flux_original_masked[1000] / flux[1000]
    normalization_factor_2 = flux_original_masked[1500] / flux[1500]
    normalization_factor_3 = flux_original_masked[2000] / flux[2000]

    if not (np.allclose([normalization_factor, normalization_factor_2, normalization_factor_3], normalization_factor, rtol = 0.001)):
        raise ValueError(f"Normalization factors disagree: {normalization_factor:.3f}, {normalization_factor_2:.3f}, {normalization_factor_3:.3f}")
    
    # f = interp1d(wave, best_fit_spec, kind = 'linear') # no need to interp2d
    # continuum_interp_values = f(wave_original[mask]) * normalization_factor

    continuum_hdu.data[mask] = best_fit_spec * normalization_factor # continuum_interp_values

    out_path = str(fits_path).replace("_x1dsum.fits", "_continuumfit.fits")
    new_hdul.writeto(str(out_path), overwrite = True)

In [7]:
reload(cfg)

d = pd.DataFrame({
    'instr': ['COS', 'COS', 'COS', 'COS', 'STIS', 'STIS', 'Gaussian'],
    'gratings': ['G130M', 'G160M', 'G185M', 'G225M', 'E230M', 'E140M', 'N/A'],
    'slits': ['NA', 'NA', 'NA', 'NA', '0.2x0.2', '0.2x0.06', 'NA'],
    'lsfranges': [[1100, 1460], [1400, 1800], [1800, 2100], [2100, 2278], [1607, 3129], [1144, 1729], [899, 1191]],
    'lps': ['1', '1', '1', '1', '1', '1', 'NA'],
    'cen_wave': ['1291', '1611', '1921', '2250', '1978', '1425', 'NA'],
    'pixel_scales': [None, None, None, None, None, None, 0.013], 
    'fwhms': [None, None, None, None, None, None, 0.02]
})

num = 0

cfg.lsfs = []
for col in ['instr', 'gratings', 'slits', 'cen_wave', 'pixel_scales', 'fwhms']:
    setattr(cfg, col, [d.at[num, col]])

cfg.lsfranges = np.array([d.at[num, 'lsfranges']])

In [8]:
for spectra_directory in directories:

    current_directory = os.path.join(base_directory, spectra_directory)

    os.chdir(current_directory)

    igm_model = glob.glob(os.path.join(current_directory, "*_x1dfits_model.json"))[0]
    continuumfit = glob.glob(os.path.join(current_directory, "*_continuumfit.fits"))[0]

    utils.pyigm_to_veeper(igm_model, continuumfit)

    os.chdir(base_directory)

os.chdir(notebook_directory)

Loading abundances from Asplund2009
Abundances are relative by number on a logarithmic scale with H=12


In [9]:
for spectra_directory in directories:

    current_directory = os.path.join(base_directory, spectra_directory)

    component_groups_dir = os.path.join(current_directory, "component_groups")
    output_dir = os.path.join(current_directory, "modified_component_groups")

    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    mc_lines_file = "/Users/billyli/Documents/float-for-morrow/LMC/lmc_g130m1291.txt"
    mc_lines_df = pd.read_csv(mc_lines_file, sep = r"\s+", header = None, names = ["line", "wavelength"])
    mc_lines_df["wavelength"] = mc_lines_df["wavelength"].astype(float)

    def is_valid_line(row, line_df):
        trans = row["trans"].strip()
        restwave = row["restwave"]
        matching_lines = line_df[line_df["line"] == trans]
        for wave in matching_lines["wavelength"]:
            if 0.999 * wave <= restwave <= 1.001 * wave:
                return True
        return False

    input_files = glob.glob(os.path.join(component_groups_dir, "*.txt"))

    for file in input_files:
        df = pd.read_csv(file, sep = "|")
        df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
    
        df["restwave"] = pd.to_numeric(df["restwave"], errors = "coerce")
        df["bval"] = pd.to_numeric(df["bval"], errors = "coerce")
    
        valid_mask = df.apply(lambda row: is_valid_line(row, mc_lines_df), axis = 1)
        df_filtered = df[valid_mask].copy()
    
        # df_filtered["col"] = df_filtered["col"] - 0.2
        df_filtered["bval"] = df_filtered["bval"] / 2
    
        output_file = os.path.join(output_dir, os.path.basename(file))
        df_filtered.to_csv(output_file, sep = "|", index = False)
        # print(f"Processed file: {os.path.basename(file)} -> {output_file}")

In [10]:
# Before fitting, should manually adjust the input group files

In [11]:
for spectra_directory in directories:

    current_directory = os.path.join(base_directory, spectra_directory)

    os.chdir(current_directory)

    igm_models = glob.glob(os.path.join(current_directory, "*_x1dfits_model.json"))
    continuumfits = glob.glob(os.path.join(current_directory, "*_continuumfit.fits"))

    component_groups_dir = os.path.join(current_directory, "modified_component_groups")
    
    input_group_files = sorted([f for f in glob.glob(os.path.join(component_groups_dir, "input_group_*.txt")) if re.match(r".*input_group_\d+\.txt$", f)])

    VPmeasure.batch_fit(continuumfits[0], input_group_files, filepath = './modified_component_groups/')

    os.chdir(base_directory)

os.chdir(notebook_directory)

Bad METADATA;  proceeding without
`ftol` termination condition is satisfied.
Function evaluations 16, initial cost 1.9111e+03, final cost 4.7249e+02, first-order optimality 2.70e-03.

Fit results: 

1142.37	 0.000020	 14.699	 15.534	 -3.3491
 	  	  	 0.025	 0.635	 0.539 

1143.23	 0.000020	 14.699	 15.534	 -3.3491
 	  	  	 0.0	 0.0	 0.0 

1144.94	 0.000020	 14.699	 15.534	 -3.3491
 	  	  	 0.0	 0.0	 0.0 

1142.37	 0.000900	 15.077	 24.468	 -0.76774
 	  	  	 0.016	 0.528	 0.479 

1143.23	 0.000900	 15.077	 24.468	 -0.76774
 	  	  	 0.0	 0.0	 0.0 

1144.94	 0.000900	 15.077	 24.468	 -0.76774
 	  	  	 0.0	 0.0	 0.0 

1250.58	 0.000900	 15.577	 22.170	 6.6679
 	  	  	 0.011	 0.495	 0.343 

1253.8	 0.000900	 15.577	 22.170	 6.6679
 	  	  	 0.0	 0.0	 0.0 


Reduced chi-squared: 2.014878
Iteration 1 -
`ftol` termination condition is satisfied.
Function evaluations 16, initial cost 1.9111e+03, final cost 4.7249e+02, first-order optimality 2.70e-03.

Fit results: 

1142.37	 0.000020	 14.699	 15

In [12]:
for spectra_directory in directories:

    current_directory = os.path.join(base_directory, spectra_directory)

    os.chdir(current_directory)

    df = pd.read_csv('compiledVPoutputs.dat', sep = '|')
    filtered_df = df[df['zsys'] > 0.0002]
    filtered_df = filtered_df[filtered_df['sigcol'] != 0.000]
    result = filtered_df[['col', 'sigcol', 'trans']]

    unique_result = result#.drop_duplicates()
    print(unique_result)
    os.chdir(base_directory)

os.chdir(notebook_directory)

       col  sigcol trans
0   15.963   0.025  MgII
2   13.499   0.038  NiII
4   12.215   0.196  CuII
8   15.077   0.016  FeII
11  15.577   0.011  S II
13  13.765   0.020  P II


In [13]:
directories = [d for d in os.listdir(base_directory)
               if os.path.isdir(os.path.join(base_directory, d))]

records = []

for specdir in directories:
    fn = os.path.join(base_directory, specdir, "compiledVPoutputs.dat")
    if not os.path.isfile(fn):
        continue

    df = pd.read_csv(fn, sep = "|")
    df = df[(df["zsys"] > 0.0002) & (df["sigcol"] != 0.0)]
    records.append(df[["trans", "col", "sigcol"]])

all_measurements = pd.concat(records, ignore_index = True)

def random_effects_mean(vals, sigmas):
    vals = np.asarray(vals, dtype = float)
    sigmas = np.asarray(sigmas, dtype = float)
    var = sigmas ** 2

    w  = 1.0 / var
    mu = (w * vals).sum() / w.sum()

    Q  = (w * (vals - mu) ** 2).sum()
    df = len(vals) - 1
    c  = w.sum() - (w ** 2).sum() / w.sum()
    T2 = max(0.0, (Q - df) / c) if c > 0 else 0.0

    w_re = 1.0 / (var + T2)
    mean = (w_re * vals).sum() / w_re.sum()
    std  = np.sqrt(1.0 / w_re.sum())
    return mean, std

combined = (
    all_measurements
    .groupby("trans", sort = True)
    .apply(lambda g: random_effects_mean(g["col"], g["sigcol"]))
    .apply(pd.Series)
    .reset_index()
    .rename(columns = {0: "col_combined", 1: "sigcol_combined"})
)

combined = combined.sort_values("trans").reset_index(drop = True)

for _, row in combined.iterrows():
    print(f"{row.trans}  {row.col_combined:.3f} ± {row.sigcol_combined:.3f}")

output_file = os.path.join(notebook_directory, "abundances_g130m1291.txt")

with open(output_file, "w") as f:
    for _, row in combined.iterrows():
        f.write(f"{row.trans} {row.col_combined:.3f} {row.sigcol_combined:.3f}\n")

CuII  12.215 ± 0.196
FeII  15.077 ± 0.016
MgII  15.963 ± 0.025
NiII  13.499 ± 0.038
P II  13.765 ± 0.020
S II  15.577 ± 0.011


/var/folders/7n/v6gcxcpj68q6nnv2znnc85xc0000gn/T/ipykernel_6720/874184323.py:38: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: random_effects_mean(g["col"], g["sigcol"]))


In [14]:
# END